# 1. Setup and Imports libraries

In [29]:
# 1. Import libraries
import os
import requests
import datetime
import time
import json
import pandas as pd
import urllib.parse

# 2. Setup Session
session = requests.Session()

# Bắt buộc phải có User-Agent cụ thể để Reddit không block (Lỗi 429)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json'
}
session.headers.update(headers)

print("✅ Session created with headers!")

✅ Session created with headers!


In [30]:
# Define BASE_DIR if not already set
BASE_DIR = globals().get("BASE_DIR", os.getcwd())
BASE_DIR = r"D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results"

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)
    print(f"📁 Đã setup thư mục mới tại: {BASE_DIR}")
else:
    print(f"✅ Thư mục đã sẵn sàng tại: {BASE_DIR}")

# (Tùy chọn) In ra đường dẫn tuyệt đối để bạn dễ dàng biết chính xác file đang lưu ở đâu trên máy
print("Absolute Output folder:", os.path.abspath(BASE_DIR))

✅ Thư mục đã sẵn sàng tại: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results
Absolute Output folder: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results


# 2. Helper functions

In [31]:
def convert_utc_to_vn(timestamp):
    """Chuyển đổi timestamp sang múi giờ Việt Nam"""
    if not timestamp:
        return None
    dt_utc = pd.to_datetime(timestamp, unit='s', utc=True)
    dt_vn = dt_utc.tz_convert("Asia/Ho_Chi_Minh").tz_localize(None)
    return dt_vn

def extract_comments(comment_list, parent_id=None):
    """Hàm đệ quy bóc tách toàn bộ comment và reply từ JSON của Reddit"""
    extracted = []
    
    for item in comment_list:
        # Bỏ qua các object loại 'more' (nút "tải thêm bình luận" của Reddit)
        if item.get('kind') == 'more':
            continue
            
        data = item.get('data', {})
        
        # Bỏ qua nếu comment bị xóa hoặc tác giả bị xóa
        if data.get('author') == '[deleted]' or data.get('body') == '[removed]':
            continue
            
        cid = data.get('id')
        
        # Tạo record cho comment hiện tại
        comment_info = {
            'comment_id': cid,
            'parent_id': parent_id,
            'is_reply': 1 if parent_id else 0,
            'author': data.get('author'),
            'text': data.get('body'),
            'upvotes': data.get('ups'),
            'created_at_vn': convert_utc_to_vn(data.get('created_utc'))
        }
        extracted.append(comment_info)
        
        # Kiểm tra xem comment này có reply (con) không
        replies = data.get('replies')
        if replies and isinstance(replies, dict):
            # Nếu có, tiếp tục gọi đệ quy để lấy các comment con
            child_comments = replies.get('data', {}).get('children', [])
            extracted.extend(extract_comments(child_comments, parent_id=cid))
            
    return extracted


def get_reddit_links_by_keyword(keyword, session, max_links=50):
    """
    Tìm kiếm từ khóa trên Reddit và trả về danh sách URL bài viết.
    Sử dụng tham số 'after' của Reddit để phân trang (lật trang).
    """
    encoded_kw = urllib.parse.quote_plus(keyword)
    links_found = []
    after = None  # Token để Reddit biết đang ở trang nào
    
    print(f"🔍 Bắt đầu quét Reddit cho từ khóa: '{keyword}'...")
    
    while len(links_found) < max_links:
        # URL tìm kiếm API của Reddit (thêm tham số after để sang trang tiếp theo)
        search_url = f"https://www.reddit.com/search.json?q={encoded_kw}&limit=25"
        if after:
            search_url += f"&after={after}"
            
        try:
            res = session.get(search_url, timeout=15)
            
            if res.status_code != 200:
                print(f"❌ Lỗi HTTP {res.status_code} khi tìm kiếm.")
                break
                
            data = res.json()
            posts = data['data']['children']
            
            if not posts:
                print("   ⚠️ Đã hết kết quả tìm kiếm.")
                break
                
            for post in posts:
                # Lấy đường dẫn (permalink) của bài viết
                permalink = post['data']['permalink']
                full_url = f"https://www.reddit.com{permalink}"
                
                if full_url not in links_found:
                    links_found.append(full_url)
                    
                if len(links_found) >= max_links:
                    break
            
            # Lấy token 'after' để dùng cho lần lặp (trang) tiếp theo
            after = data['data'].get('after')
            if not after:
                break # Không còn trang nào nữa
                
            print(f"   -> Đã gom được {len(links_found)} link. Đang lật sang trang tiếp theo...")
            time.sleep(1.5) # Tránh bị rate limit
            
        except Exception as e:
            print(f"❌ Lỗi khi tìm kiếm: {e}")
            break
            
    print(f"✅ HOÀN TẤT: Tìm thấy tổng cộng {len(links_found)} bài viết.")
    return links_found

# 3. Function to crawl a single Reddit post

In [32]:
def crawl_reddit_post(post_url, session, sleep=1.5):
    """
    Cào dữ liệu của 1 bài viết và comments.
    Reddit hỗ trợ xuất JSON bằng cách thêm .json vào cuối URL.
    """
    # Xử lý URL (đảm bảo đuôi .json)
    if not post_url.endswith('.json'):
        post_url = post_url.rstrip('/') + '.json'
        
    try:
        time.sleep(sleep) # Tránh bị rate limit
        response = session.get(post_url, timeout=15)
        
        if response.status_code != 200:
            print(f"❌ HTTP Lỗi {response.status_code} tại URL: {post_url}")
            return None, None
            
        json_data = response.json()
        
        # JSON Reddit bài viết luôn trả về 1 list gồm 2 phần tử:
        # [0] là thông tin Bài viết, [1] là danh sách Comments
        post_data = json_data[0]['data']['children'][0]['data']
        comments_data = json_data[1]['data']['children']
        
        # 1. Bóc tách Metadata bài viết
        post_info = {
            'post_id': post_data.get('id'),
            'title': post_data.get('title'),
            'author': post_data.get('author'),
            'subreddit': post_data.get('subreddit'),
            'upvotes': post_data.get('ups'),
            'num_comments': post_data.get('num_comments'),
            'created_at_vn': convert_utc_to_vn(post_data.get('created_utc')),
            'url': post_data.get('url') # Link ảnh/bài viết gốc nếu có
        }
        
        # 2. Bóc tách Comments
        all_comments = extract_comments(comments_data)
        
        print(f"✅ Đã cào thành công bài: {post_info['title'][:50]}... | Lấy được {len(all_comments)} comments.")
        return post_info, all_comments
        
    except Exception as e:
        print(f"⚠️ Lỗi cào dữ liệu bài viết: {e}")
        return None, None

# 4. Save to CSV files


In [33]:
def export_reddit_posts_to_csv(all_posts, out_dir):
    """Export post metadata to CSV"""
    os.makedirs(out_dir, exist_ok=True)
    
    rows = []
    for post in all_posts:
        rows.append({
            'post_id': post.get('post_id'),
            'title': post.get('title'),
            'author': post.get('author'),
            'subreddit': post.get('subreddit'),
            'upvotes': post.get('upvotes'),
            'num_comments': post.get('num_comments'),
            'created_at_vn': post.get('created_at_vn'),
            'url': post.get('url')
        })
    
    df = pd.DataFrame(rows)
    csv_path = os.path.join(out_dir, "reddit_posts_metadata.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ Saved posts metadata CSV: {csv_path}")
    return csv_path, df


def export_reddit_all_comments_to_csv(all_comments, out_dir):
    """Export all comments to CSV"""
    os.makedirs(out_dir, exist_ok=True)
    
    df = pd.DataFrame(all_comments)
    # Reorder columns for better readability
    cols = ['post_id', 'comment_id', 'parent_id', 'is_reply', 'author', 'text', 'upvotes', 'created_at_vn']
    df = df[[c for c in cols if c in df.columns]]
    
    csv_path = os.path.join(out_dir, "all_comments.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ Saved comments CSV: {csv_path}")
    return csv_path, df


# 5. Main function to crawl multiple Reddit posts

In [34]:
# ==========================================
# CẤU HÌNH TOOL REDDIT
# ==========================================
KEYWORD = "deep learning"
MAX_POSTS = 20

print(f"{'='*60}")
print(f"🚀 BẮT ĐẦU CÀO REDDIT TỰ ĐỘNG TỪ KHÓA: {KEYWORD.upper()}")
print(f"{'='*60}\n")

# 1. TỰ ĐỘNG LẤY LINK
urls_to_crawl = get_reddit_links_by_keyword(KEYWORD, session, max_links=MAX_POSTS)

# 2. TIẾN HÀNH CÀO TỪNG BÀI
all_posts = []
all_comments_data = []

for idx, url in enumerate(urls_to_crawl, 1):
    print(f"\n[{idx}/{len(urls_to_crawl)}] Xử lý bài...")
    
    post_info, comments = crawl_reddit_post(url, session=session)
    
    if post_info:
        # Lưu post info vào list
        all_posts.append(post_info)
        
        # In thử vài comment ra màn hình
        if comments:
            print("   💬 Trích xuất thử một vài bình luận:")
            for cmt in comments[:3]: 
                short_text = cmt['text'][:60].replace('\n', ' ') + "..." if len(cmt['text']) > 60 else cmt['text'].replace('\n', ' ')
                print(f"      👤 {cmt['author']}: {short_text} (👍 {cmt['upvotes']})")
            
            # Thêm post_id vào mỗi comment
            for cmt in comments:
                cmt['post_id'] = post_info['post_id']
                all_comments_data.append(cmt)
            
            print(f"📊 Extracted {len(comments)} comments from this post")
        else:
            print("   ⚠️ Bài viết này không có bình luận nào.")
        
    print("-" * 60)

# ==========================================
# EXPORT CSV FILES
# ==========================================
print(f"\n{'='*60}")
print("📁 EXPORTING CSV FILES...")
print(f"{'='*60}")

# Export all posts metadata to one CSV
if all_posts:
    posts_csv_path, df_posts = export_reddit_posts_to_csv(all_posts, BASE_DIR)
    print(f"📄 Saved {len(df_posts)} posts metadata")

# Export all comments to one CSV
if all_comments_data:
    comments_csv_path, df_comments = export_reddit_all_comments_to_csv(all_comments_data, BASE_DIR)
    print(f"💬 Saved {len(df_comments)} total comments")

print(f"\n{'='*60}")
print("✅ HOÀN THÀNH TOÀN BỘ QUÁ TRÌNH!")
print(f"📁 All data saved to: {BASE_DIR}")
print(f"{'='*60}")


🚀 BẮT ĐẦU CÀO REDDIT TỰ ĐỘNG TỪ KHÓA: DEEP LEARNING

🔍 Bắt đầu quét Reddit cho từ khóa: 'deep learning'...
   -> Đã gom được 20 link. Đang lật sang trang tiếp theo...
✅ HOÀN TẤT: Tìm thấy tổng cộng 20 bài viết.

[1/20] Xử lý bài...
✅ Đã cào thành công bài: Is Machine Learning / Deep Learning still a good c... | Lấy được 36 comments.
   💬 Trích xuất thử một vài bình luận:
      👤 ocean_protocol: Honestly speaking, we are seeing a rapid advancement of ML m... (👍 152)
      👤 TheRealFakeWannabe: what new ML models that are being developed is going to beco... (👍 8)
      👤 Medical-Object-4322: There's no such thing as "standard" for an ML model. It all ... (👍 16)
📊 Extracted 36 comments from this post
------------------------------------------------------------

[2/20] Xử lý bài...
✅ Đã cào thành công bài: Deep Learning in HFT... | Lấy được 51 comments.
   💬 Trích xuất thử một vài bình luận:
      👤 Snakd13: Your message is a great example of confirmation bias. If Blo... (👍 67)
      👤 Alp